## Text input

https://platform.openai.com/docs/models

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import os
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent

model = init_chat_model(
    model="openai/gpt-5-nano",
    model_provider="openai",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_KEY"],
)

agent = create_agent(
    model=model,
    system_prompt="You are a science fiction writer, create a capital city at the users request.",
)

In [3]:
from langchain.messages import HumanMessage

question = HumanMessage(content=[
    {"type": "text", "text": "What is the capital of The Moon?"}
])

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

In this science-fiction setting, the Moon is a sovereign city-state and its capital is Selene Prime (often called Selene City).

- Location: perched along the rim of Shackleton Crater in the southern polar region, where solar concentrators keep the city bathed in daylight even as the polar night looms elsewhere.
- Government: seat of the Lunar Commonwealth’s leadership. The Great Council meets in the Dome of Echo, a translucent palace that hums with ionized light.
- Architecture: a blend of basalt towers, glassy domes, and latticework tunnels carved into the lunar rock. Structures are shielded with regolith and powered by vast solar arrays and a closed-loop life support system.
- Landmarks: the Helio-Archive (crystal library of lunar science and history), the Prism Gate (the ceremonial entrance to the city), and the Lunar Opera House (where gravity is adjusted for performances).
- Economy & daily life: a hub for water-ice mining, solar manufacturing, and vertical farming. Gravity is ge

## Image input

In [4]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [6]:
print(uploader.value)

({'name': 'Generated Image August 21, 2026 - 7_17PM.png', 'type': 'image/png', 'size': 2227762, 'content': <memory at 0x117ca5840>, 'last_modified': datetime.datetime(2026, 8, 21, 11, 17, 13, 181000, tzinfo=datetime.timezone.utc)},)


In [7]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [9]:
multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this picture"},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

This picture feels like a moment from a spacefaring saga. A colossal crystalline warrior or guardian floats above a storm-washed void, its armor a jagged crown of violet shards that glow with inner light. Lightning tears across a furious purple sky, and fragments of ancient structures drift in the vacuum like fallen stars. Far below, a blue-green planet curves on the horizon, while ruined columns and temples drift in the debris—as if a once-great city hangs suspended in the rings of a shattered world. The whole scene has a ceremonial, cataclysmic grandeur—as if the guardian is ascending to defend, or to be judged, by a powerful cosmos.

Capital city concept inspired by this image:
Name: Crystalis Prime
Role: Capital of the Shard Dominion, a civilization that thrives on harnessing storms and crystal magic.

Setting and feel:
- Location: A ring of colossal, broken arches and crystal towers orbiting a brilliant blue planet. The city is built into and around floating crystal fragments, tet

## Audio input

In [ ]:
import sounddevice as sd
from scipy.io.wavfile import write
import base64
import io
import time
from tqdm import tqdm

# Recording settings
duration = 5  # seconds
sample_rate = 44100

print("Recording...")
audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1)
# Progress bar for the duration
for _ in tqdm(range(duration * 10)):   # update 10× per second
    time.sleep(0.1)
sd.wait()
print("Done.")

# Write WAV to an in-memory buffer
buf = io.BytesIO()
write(buf, sample_rate, audio)
wav_bytes = buf.getvalue()

aud_b64 = base64.b64encode(wav_bytes).decode("utf-8")

In [ ]:
audio_model = init_chat_model(
    model="openai/gpt-audio",
    model_provider="openai",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_KEY"],
)

agent = create_agent(
    model=audio_model,
)

multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this audio file"},
    {"type": "audio", "base64": aud_b64, "mime_type": "audio/wav"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)